# Array

Used to store multiple items of same type in continous memory locations.

## **Drawback**
- Fixed size which can lead to memory wasteage
- Homogenous data which lacks flexibility

In [ ]:
import numpy as np

# 1. FIXED TYPE
print("--- Fixed Type ---")

# Create an array explicitly typed as integers
int_array = np.array([1, 2, 3, 4], dtype=int)
print(f"Original integer array: {int_array}")

# Try assigning a float (7.9) into the integer array
int_array[0] = 7.9
print(f"After assigning 7.9 to index 0: {int_array}")
print("Notice that 7.9 was silently truncated to 7 because the array stores raw ints, not objects.\n")

In [ ]:
# 2. INSERT MAKES A COPY
print("--- Insert Makes a Copy ---")
a = np.array([10, 20, 30, 40])
print(f"Original 'a': {a} (Memory ID: {id(a)})")

# Insert 99 at index 2
new_array = np.insert(a, 2, 99)

print(f"Returned 'new_array': {new_array} (Memory ID: {id(new_array)})")
print(f"Original 'a' after insert: {a}")
print("Because 'a' is a fixed-size contiguous block, it cannot expand. NumPy had to create a fresh copy.")

## Homogenity problem - **Referential arrays**
- Python List is example of Referential array
- Data stored in different locations but their reference is stored in continous memory locations -- array of references

In [ ]:
# Create a standard Python list
python_list = [10, 20, 30, 40]

print("--- Python List Element Memory Locations ---")
for i, element in enumerate(python_list):
    # Get the memory address of the actual element
    address = id(element)

    # Convert it to hexadecimal (how computer memory addresses are usually written)
    hex_address = hex(address)

    print(f"Index {i} (Value: {element}) -> Memory Address: {address} ({hex_address})")

##**Drawback**: referential arrays
- extra memory
- bit slower incomparision with arrays


## Fixed size problem?
**Dynamic array**
- array gets adjusted depending on data
- Python List is an example of Dynamic array


In [ ]:
import sys
list1 = []
sys.getsizeof(list1) # in ram how many bytes its occupying.

In [ ]:
list1.append("Hello")
print(list1)
sys.getsizeof(list1) # you can see its increased

In [ ]:
list1.append("world")
print(list1)
sys.getsizeof(list1) # It remains same as last

In [ ]:
for i in range(50):
    print(i, sys.getsizeof(list1))
    list1.append(i)

### What the output above shows
- An empty list is **56 bytes**. This is only the list's header (length, capacity and a pointer to the block of references), with no slots yet.
- Each slot is **8 bytes**, because it holds a reference (memory address), not the actual value. So `capacity = (size - 56) / 8`.
- `"Hello"` and `"world"` did not add their own size. `getsizeof` only counts the references, which shows the list is a **referential array**.
- The size does **not** grow on every `append`. Python keeps some extra empty slots (**over-allocation**). When they run out, it:
  1. creates a bigger block of memory,
  2. copies all the references into it,
  3. frees the old block.
- Capacity grows 4 → 8 → 16 → 24 → 32 → 40 → 52 → 64 ... (about 12.5% extra each time). Because resizing is rare, `append` is **O(1) on average (amortised)**.

In [ ]:
# 3. LENGTH vs CAPACITY
print("--- Length vs Capacity ---")
empty_size = sys.getsizeof([])   # list header with zero slots
list2 = []
last_size = sys.getsizeof(list2)

for i in range(60):
    list2.append(i)
    size = sys.getsizeof(list2)
    if size != last_size:   # print only when Python resized the list
        capacity = (size - empty_size) // 8   # each slot is an 8-byte reference
        print(f"len = {len(list2):>2} -> resized to {size} bytes (capacity {capacity})")
        last_size = size

##**Advantage of Array:**

Indexing operation is faster in Array -
For example let us say we have a list with 50000 items. and we want to access 5000th item then we use index `list1[4999]`

Under the hood:

`Address to access = base address + index * datatype size`

i.e., Python looks at the list1's base address. It calculates the location of the 4999 reference using your exact math: Base + 4999 * 8 bytes

##**Disadvantage of Array:**
- Insert and Delete Operation

### Checking the address formula
A NumPy array lets us see the real memory address of each element, so we can check `base address + index * datatype size`.

In [ ]:
# 4. ADDRESS = BASE + INDEX * SIZE
print("--- Address = Base + Index * Size ---")
arr = np.array([10, 20, 30, 40, 50], dtype=np.int64)
base = arr.ctypes.data       # memory address of arr[0]
size = arr.itemsize          # 8 bytes per int64

for i in range(len(arr)):
    calculated = base + i * size
    actual = arr[i:].ctypes.data   # real address where arr[i] starts
    print(f"Index {i}: calculated {calculated}, actual {actual}, match = {calculated == actual}")

In [ ]:
# 5. INDEXING IS O(1)
import timeit

print("--- Indexing Speed (50,000 items) ---")
list3 = list(range(50000))

# Accessing the first, middle or last item takes the same time,
# because the address is calculated directly - no walking through the list.
for index in [0, 4999, 25000, 49999]:
    t = timeit.timeit(lambda: list3[index], number=100000)
    print(f"list3[{index}] x 100000 -> {t:.4f} seconds")

### Why Insert and Delete are slow
- `insert(0, x)` has to shift **every** element one slot to the right to make space → **O(n)**
- `pop(0)` has to shift **every** element one slot to the left to fill the gap → **O(n)**
- `append(x)` and `pop()` only touch the last slot → **O(1)**

In [ ]:
# 6. INSERT / DELETE COST
print("--- Insert / Delete Speed (100,000 items) ---")
n = 1000

big = list(range(100000))
t_append = timeit.timeit(lambda: big.append(0), number=n)
t_insert = timeit.timeit(lambda: big.insert(0, 0), number=n)
print(f"append at end     x {n}: {t_append:.6f} seconds")
print(f"insert at index 0 x {n}: {t_insert:.6f} seconds  ({t_insert / t_append:.0f}x slower)")

big = list(range(100000))
t_pop_end = timeit.timeit(lambda: big.pop(), number=n)
t_pop_front = timeit.timeit(lambda: big.pop(0), number=n)
print(f"pop from end      x {n}: {t_pop_end:.6f} seconds")
print(f"pop from index 0  x {n}: {t_pop_front:.6f} seconds  ({t_pop_front / t_pop_end:.0f}x slower)")